# Cutoff Selection

Choosing the seven forecast origins the whole evaluation rests on: **4 event-driven
and 3 quiet**.

The point of the split is to separate two questions. On event cutoffs, does reading
news let an agent anticipate a shock a statistical baseline cannot see? On quiet
cutoffs, does the agent *avoid damaging* a forecast when there is nothing to react to?
A method that only wins on shocks and loses on calm weeks is not useful.

Everything here is derived from data — no dates are asserted by hand.

Prerequisites: [`01_MPOB_pko_data_exploration.ipynb`](01_MPOB_pko_data_exploration.ipynb).


---
## 1. Setup


In [1]:
from __future__ import annotations

import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv


ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(ROOT / "implementations"))
load_dotenv(ROOT / ".env")

from cpo.data import MPOB_WEEKLY_SERIES_ID, build_mpob_service
from cpo.plots import DEFAULT_CUTOFFS, HORIZONS_WEEKS, plot_cutoff_windows, plot_price_history


svc = build_mpob_service(cache_dir=ROOT / "data" / "mpob")
as_of = datetime.now(tz=timezone.utc).replace(tzinfo=None)

weekly = svc.get_series(MPOB_WEEKLY_SERIES_ID, as_of=as_of).set_index("timestamp")["value"]
returns = weekly.pct_change() * 100

articles = pd.read_csv(ROOT / "implementations" / "pko" / "palm_articles_daily.csv", parse_dates=["date"])
news_weekly = articles.set_index("date")["article_count"].resample("W-FRI").sum()

MAX_HORIZON = max(HORIZONS_WEEKS)
print(f"{len(weekly)} weekly prices, {weekly.index.min():%Y-%m-%d} -> {weekly.index.max():%Y-%m-%d}")
print(f"horizons: {HORIZONS_WEEKS} weeks (longest {MAX_HORIZON})")


971 weekly prices, 2008-01-04 -> 2026-08-07
horizons: [1, 2, 4, 8, 13] weeks (longest 13)


---
## 2. The four constraints

| # | Constraint | Why |
|---|---|---|
| 1 | Cutoff ≥ 2024-02 | GDELT news starts 2024-01; leave 4 weeks of prior context |
| 2 | Every horizon resolves | A 13-week horizon needs 13 weeks of realised prices after the cutoff |
| 3 | ≥ 100 articles in the prior 8 weeks | An agent needs news to reason over, or the comparison is empty |
| 4 | ≥ 10 weeks between cutoffs | Non-overlapping windows keep the seven scores independent |

Constraint 3 matters more than it looks — GDELT coverage collapses from mid-2025 to
early 2026, and a cutoff there would test nothing.


In [2]:
def summarise(cutoff: pd.Timestamp) -> dict | None:
    """Return forward-looking stats for a candidate cutoff, or None if it cannot resolve."""
    targets = [cutoff + pd.Timedelta(weeks=h) for h in HORIZONS_WEEKS]
    if any(t not in weekly.index for t in targets):
        return None
    forward = returns.loc[cutoff + pd.Timedelta(weeks=1) : cutoff + pd.Timedelta(weeks=MAX_HORIZON)]
    return {
        "cutoff": cutoff,
        "price": float(weekly[cutoff]),
        "max_move": float(forward.abs().max()),
        "vol": float(forward.std()),
        "total_13wk": float(weekly[targets[-1]] / weekly[cutoff] - 1) * 100,
        "news_8wk": float(news_weekly.loc[cutoff - pd.Timedelta(weeks=8) : cutoff].sum()),
    }


candidates = pd.DataFrame([s for c in weekly.index if c >= pd.Timestamp("2024-02-01") and (s := summarise(c))])
eligible = candidates[candidates.news_8wk >= 100].copy()

print(f"weeks in the GDELT window        : {(weekly.index >= '2024-02-01').sum()}")
print(f"...that resolve at every horizon : {len(candidates)}")
print(f"...with enough news coverage     : {len(eligible)}")
print(f"latest usable cutoff             : {candidates.cutoff.max():%Y-%m-%d}")

weeks in the GDELT window        : 132
...that resolve at every horizon : 119
...with enough news coverage     : 79
latest usable cutoff             : 2026-05-08


---
## 3. Event cutoffs

Find the largest weekly moves, then place a cutoff **two weeks before** each one — so
the shock lands inside the forecast window rather than in the visible history. A model
that could see the move already has the answer.

Taking the four largest *distinct* moves, spread across the period, in both directions.


In [3]:
biggest = returns["2024-01-01":].copy()
biggest = biggest.reindex(biggest.abs().sort_values(ascending=False).index).head(10)
pd.DataFrame(
    {
        "week_ending": [f"{d:%Y-%m-%d}" for d in biggest.index],
        "pct_change": biggest.round(2).to_numpy(),
        "price_RM": [round(float(weekly[d])) for d in biggest.index],
    }
).head(10)

,week_ending,pct_change,price_RM
0,2024-04-19,-7.72,4100
1,2026-03-13,7.28,4492
2,2024-10-25,7.12,4686
3,2025-04-18,-7.07,4180
4,2024-12-06,6.68,5334
5,2024-12-20,-6.27,4828
6,2026-03-06,5.83,4187
7,2025-04-11,-5.59,4498
8,2024-04-05,5.29,4512
9,2025-01-03,-5.15,4726


In [4]:
EVENT_TARGETS = {
    "2024-04-19": "April 2024 selloff",
    "2024-10-25": "October 2024 rally",
    "2025-04-18": "April 2025 slide",
    "2026-03-13": "March 2026 surge",
}

grid = list(weekly.index)
event_rows = []
for week, label in EVENT_TARGETS.items():
    shock = pd.Timestamp(week)
    cutoff = grid[grid.index(shock) - 2]  # two weeks before the move
    stats = summarise(cutoff)
    event_rows.append({**stats, "shock_week": week, "shock_pct": round(float(returns[shock]), 2), "label": label})

events = pd.DataFrame(event_rows)
events.assign(cutoff=events.cutoff.dt.strftime("%Y-%m-%d"))[
    ["cutoff", "price", "shock_week", "shock_pct", "max_move", "total_13wk", "news_8wk", "label"]
].round(1)

,cutoff,price,shock_week,shock_pct,max_move,total_13wk,news_8wk,label
0,2024-04-05,4511.5,2024-04-19,-7.7,7.7,-8.9,251.0,April 2024 selloff
1,2024-10-11,4402.5,2024-10-25,7.1,7.1,7.3,158.0,October 2024 rally
2,2025-04-04,4764.5,2025-04-18,-7.1,7.1,-15.4,166.0,April 2025 slide
3,2026-02-27,3956.5,2026-03-13,7.3,7.3,13.3,45.0,March 2026 surge


Two rise, two fall, spread from April 2024 to February 2026 — so the result cannot be
an artifact of one direction or one period.

**One caveat to record:** the February 2026 cutoff has only ~45 articles in its prior
eight weeks, below our threshold, because it sits at the tail of the GDELT sparse
stretch. It is kept deliberately — it is the only 2026 event available, and a cutoff
past most LLM training windows is worth having. Expect the agent to have less to work
with there, and report it separately rather than hiding it in the average.


---
## 4. Quiet cutoffs

The calmest forward windows that clear the news threshold and stay clear of the event
cutoffs.


In [5]:
def pick_spaced(frame: pd.DataFrame, n: int, *, exclude: list[pd.Timestamp], min_gap_weeks: int = 10):
    """Greedily take the calmest windows, keeping every pick well separated."""
    chosen: list[pd.Timestamp] = []
    for _, row in frame.sort_values("max_move").iterrows():
        if any(abs((row.cutoff - e).days) < min_gap_weeks * 7 for e in exclude + chosen):
            continue
        chosen.append(row.cutoff)
        if len(chosen) == n:
            break
    return chosen


quiet_dates = pick_spaced(eligible, 3, exclude=list(events.cutoff))
quiet = eligible[eligible.cutoff.isin(quiet_dates)]
quiet.assign(cutoff=quiet.cutoff.dt.strftime("%Y-%m-%d"))[
    ["cutoff", "price", "max_move", "vol", "total_13wk", "news_8wk"]
].round(1)

,cutoff,price,max_move,vol,total_13wk,news_8wk
48,2025-01-03,4726.0,3.7,2.1,0.8,251.0
73,2025-06-27,3956.5,3.8,1.6,10.2,106.0
118,2026-05-08,4515.0,2.4,1.4,-0.1,180.0


---
## 5. The seven

Event windows should contain a visible shock; quiet windows should be flat. The
`max_move` column is the separation that makes the split meaningful.


In [6]:
selected = (
    pd.concat(
        [
            events.assign(kind="event")[["cutoff", "kind", "price", "max_move", "total_13wk", "news_8wk"]],
            quiet.assign(kind="quiet")[["cutoff", "kind", "price", "max_move", "total_13wk", "news_8wk"]],
        ]
    )
    .sort_values("cutoff")
    .reset_index(drop=True)
)
selected["cutoff"] = selected.cutoff.dt.strftime("%Y-%m-%d")
selected.round(1)

,cutoff,kind,price,max_move,total_13wk,news_8wk
0,2024-04-05,event,4511.5,7.7,-8.9,251.0
1,2024-10-11,event,4402.5,7.1,7.3,158.0
2,2025-01-03,quiet,4726.0,3.7,0.8,251.0
3,2025-04-04,event,4764.5,7.1,-15.4,166.0
4,2025-06-27,quiet,3956.5,3.8,10.2,106.0
5,2026-02-27,event,3956.5,7.3,13.3,45.0
6,2026-05-08,quiet,4515.0,2.4,-0.1,180.0


In [7]:
ev, qt = selected[selected.kind == "event"], selected[selected.kind == "quiet"]
print(f"event windows — mean max move {ev.max_move.mean():.2f}%")
print(f"quiet windows — mean max move {qt.max_move.mean():.2f}%")
print(f"separation                    : {ev.max_move.mean() / qt.max_move.mean():.1f}x")

gaps = pd.to_datetime(selected.cutoff).diff().dt.days.dropna()
print(
    f"\nclosest two cutoffs: {int(gaps.min())} days apart ({int(gaps.min()) // 7} weeks) "
    f"— windows are {'independent' if gaps.min() >= 70 else 'OVERLAPPING'}"
)

event windows — mean max move 7.30%
quiet windows — mean max move 3.31%
separation                    : 2.2x

closest two cutoffs: 70 days apart (10 weeks) — windows are independent


---
## 6. Visual check

The chart is the audit. If an orange band looks flat, or a blue band contains a cliff,
the label is wrong and the cutoff should be swapped.


In [8]:
plot_cutoff_windows(
    svc.get_series(MPOB_WEEKLY_SERIES_ID, as_of=as_of),
    cutoffs=DEFAULT_CUTOFFS,
    horizons=13,
)

In [9]:
plot_price_history(
    svc.get_series(MPOB_WEEKLY_SERIES_ID, as_of=as_of),
    cutoffs=DEFAULT_CUTOFFS,
    start="2023-06-01",
    title="MPOB weekly with the seven cutoffs",
    units="MYR per tonne",
    currency="RM",
    show_blackouts=False,
)

---
## 7. Committed selection

These are frozen in `cpo.plots.DEFAULT_CUTOFFS`, so the specs, baselines, and agent
evaluation all read the same list rather than re-deriving it.


In [10]:
pd.DataFrame(
    [{"cutoff": c.date, "kind": c.kind, "why": c.label} for c in sorted(DEFAULT_CUTOFFS, key=lambda c: c.date)]
)

,cutoff,kind,why
0,2024-04-05,event,-7.7% two weeks out; -8.9% over 13 weeks
1,2024-10-11,event,+7.1% two weeks out; +7.3% over 13 weeks
2,2025-01-03,quiet,max weekly move ahead 3.7%; flat over 13 weeks
3,2025-04-04,event,-7.1% two weeks out; -15.4% over 13 weeks
4,2025-06-27,quiet,max weekly move ahead 3.8%
5,2026-02-27,event,+7.3% two weeks out; +13.3% over 13 weeks
6,2026-05-08,quiet,"calmest window: max 2.4%, flat over 13 weeks"


---
## 8. What this does and does not establish

**Established:** seven origins on a complete weekly grid, no publication lag, no roll
artifacts, non-overlapping windows, and a 2x separation in forward volatility between
event and quiet.

**Limitations to state in any writeup:**

- **Seven origins is a small sample.** Five horizons each gives 35 scored points. Mean
  CRPS differences between close predictors will not be significant. These cutoffs are
  the narrative set; a denser weekly backtest should decide which model is actually better.
- **Events were chosen with hindsight.** We know which weeks moved. That is fine for
  a controlled comparison, but it is not a live forecasting record.
- **The 2026-02-27 cutoff is news-poor** (~45 articles) and should be reported separately.
- **No 2022-style shock is available.** The largest move in the GDELT window is ~7.7%,
  against 30%+ during the export ban. The event/quiet contrast is a matter of degree.

**Next:** a naive last-value baseline across these seven, then Prophet and the Darts
models, then the agent.
